In [ ]:
import sys
import os

# Go to the cloned repo folder
repo_path = '/content/NLP-sequence-classification'

# Add to Python path
sys.path.append(repo_path)

# Change working directory to repo
os.chdir(repo_path)

In [ ]:
from src.data_loader import get_BESSTIE_splits
import pandas as pd
from src.lr_feature_extraction import tfidf_features, load_tfidf_features
from models.logistic_regression import LogisticRegressionModel

In [ ]:
#loading and getting the splits from the dataset
df_all, df_train, df_validation,df_test = get_BESSTIE_splits()


In [ ]:
# Extract TFIDF Features
X_train, X_validation, X_test, vectorizer = tfidf_features(
    df_train, df_validation, df_test,
    text_column='text',
    max_features=15000,
    save_path="./models/tfidf"
)

In [ ]:
#train the model
model = LogisticRegressionModel()
model.train_logistic_regression(X_train, df_train)

In [ ]:
#Evaluate the model
lr_eval_results = model.evaluate_logistic_regression(X_validation, df_validation)
print("\n📌 SARCASM DETECTION:")
print(f"   Accuracy:  {lr_eval_results['Sarcasm']['accuracy']:.4f}")
print(f"   Precision: {lr_eval_results['Sarcasm']['precision']:.4f}")
print(f"   Recall:    {lr_eval_results['Sarcasm']['recall']:.4f}")
print(f"   F1-Score:  {lr_eval_results['Sarcasm']['f1']:.4f}")

print("\n📌 SENTIMENT ANALYSIS:")
print(f"   Accuracy:  {lr_eval_results['Sentiment']['accuracy']:.4f}")
print(f"   Precision: {lr_eval_results['Sentiment']['precision']:.4f}")
print(f"   Recall:    {lr_eval_results['Sentiment']['recall']:.4f}")
print(f"   F1-Score:  {lr_eval_results['Sentiment']['f1']:.4f}")


model has hit the performance ceiling for TF-IDF + Logistic Regression on this dataset. The issue is not the features or parameters - it's the model capacity

Baseline result
"The TF-IDF + Logistic Regression baseline achieves 44.4% F1 for sarcasm detection and 83.8% F1 for sentiment analysis. This establishes a lower bound for comparison with transformer-based models."


Sarcasm --- 	Requires understanding context, not just keywords
Dataset Only 14% sarcastic overall - huge class imbalance
en-IN/en-UK have 7% sarcastic ---
	Very few examples to learn from
TF-IDF limitations	Cannot capture negation, irony, or pragmatic meaning


In [ ]:
# Save model
model.save_model("./models/lr_model.pkl")

In [ ]:
# FINAL TEST EVALUATION
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
test_predictions = model.prediction_logistic_regression(X_test)

sarcasm_true_labels = df_test['Sarcasm'].astype(int).values
sentiment_true_labels = df_test['Sentiment'].astype(int).values

sarcasm_f1 = f1_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_precision = precision_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_recall = recall_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_accuracy = accuracy_score(sarcasm_true_labels, test_predictions['Sarcasm'])

sentiment_f1 = f1_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_precision = precision_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_recall = recall_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_accuracy = accuracy_score(sentiment_true_labels, test_predictions['Sentiment'])


print("FINAL TEST RESULTS")

print("\n📌 SARCASM DETECTION:")
print(f"   Accuracy:  {sarcasm_accuracy:.4f}")
print(f"   Precision: {sarcasm_precision:.4f}")
print(f"   Recall:    {sarcasm_recall:.4f}")
print(f"   F1-Score:  {sarcasm_f1:.4f}")

print("\n📌 SENTIMENT ANALYSIS:")
print(f"   Accuracy:  {sentiment_accuracy:.4f}")
print(f"   Precision: {sentiment_precision:.4f}")
print(f"   Recall:    {sentiment_recall:.4f}")
print(f"   F1-Score:  {sentiment_f1:.4f}")

Validation Results -- tried fine tuning with multiple configs


Original (ngram=1,2; C=1.0; balanced class weights)

sarcasm acc:44.4%

sentiment acc: 83.8%	==> Best sentiment performance

Trigrams (ngram=1,3)

sarcasm acc: 44.1%

sentiment acc: 84.1%	==> Slightly better sentiment, similar sarcasm

Custom weights (1:3.57)	---> Made both tasks worse

sarcasm acc: 39.6%

sentiment acc: 74.7%

Test results

Original (ngram=1,2; C=1.0; balanced)

sarcasm acc: 40.7%

sentiment acc: 81.2%